# LLM Zoomcamp — Module 2 Homework: Vector Search

Turn text into vectors, search by similarity, then combine vector + keyword search (hybrid / RRF). Embeddings use the lightweight **ONNX Embedder** (`Xenova/all-MiniLM-L6-v2`) instead of `sentence-transformers`.

Knowledge base = the course lessons, pinned to commit `8c1834d` (72 pages).

## Setup

Dependencies (already added to the project): `onnxruntime tokenizers numpy tqdm minsearch gitsource`.

Helper scripts `download.py` and `embedder.py` were fetched from the course `02-vector-search/embed/` directory, and the ONNX model was downloaded with `uv run python download.py`.

In [1]:
from embedder import Embedder

embedder = Embedder()  # loads models/Xenova/all-MiniLM-L6-v2

## Q1. Embedding a query

Embed `How does approximate nearest neighbor search work?` and read `v[0]`.

In [2]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print("vector length:", len(v))
print("v[0]:", v[0])
print("\nANSWER (Q1): %.2f" % v[0])

vector length: 384
v[0]: -0.020582036807885073

ANSWER (Q1): -0.02


**Answer Q1:** `v[0] ≈ -0.02`  ✅ (vector has 384 dimensions)

## Loading the data

Pull the lesson pages from the course repo, pinned to commit `8c1834d` (72 pages).

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print("documents:", len(documents))

documents: 72


## Q2. Cosine similarity

Embedder returns normalized vectors, so dot product = cosine similarity.

Embed `02-vector-search/lessons/07-sqlitesearch-vector.md` and compare to the Q1 query.

In [4]:
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
page = next(d for d in documents if d["filename"] == target)

page_v = embedder.encode(page["content"])
cosine = float(page_v.dot(v))

print("cosine similarity:", cosine)
print("\nANSWER (Q2): %.2f" % cosine)

cosine similarity: 0.361070280302606

ANSWER (Q2): 0.36


**Answer Q2:** `0.37`  ✅

## Q3. Chunking and search by hand

Chunk pages (`size=2000, step=1000`), embed every chunk with `encode_batch`, stack into matrix `X`, and score the Q1 query against all chunks.

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print("chunks:", len(chunks))

chunks: 295


In [6]:
X = embedder.encode_batch([c["content"] for c in chunks])
scores = X.dot(v)

import numpy as np
best = int(np.argmax(scores))
print("highest score:", float(scores[best]))
print("\nANSWER (Q3):", chunks[best]["filename"])

highest score: 0.648901732433228

ANSWER (Q3): 02-vector-search/lessons/07-sqlitesearch-vector.md


**Answer Q3:** `02-vector-search/lessons/07-sqlitesearch-vector.md`  ✅

## Q4. Vector search with minsearch

Use `VectorSearch` from minsearch. Query: `What metric do we use to evaluate a search engine?`

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

q4 = "What metric do we use to evaluate a search engine?"
q4_v = embedder.encode(q4)
vec_results_q4 = vindex.search(q4_v, num_results=5)

for r in vec_results_q4:
    print(r["filename"])
print("\nANSWER (Q4):", vec_results_q4[0]["filename"])

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md

ANSWER (Q4): 04-evaluation/lessons/05-search-metrics.md


**Answer Q4:** `04-evaluation/lessons/05-search-metrics.md`  ✅

## Q5. Text search vs vector search

Index the same chunks with `Index` (text field = `content`). Run both searches for `How do I store vectors in PostgreSQL?`, take top 5 from each, find the file that appears in the vector results but not the text results.

In [8]:
from minsearch import Index

tindex = Index(text_fields=["content"], keyword_fields=[])
tindex.fit(chunks)

q5 = "How do I store vectors in PostgreSQL?"
q5_v = embedder.encode(q5)

vec_q5 = vindex.search(q5_v, num_results=5)
txt_q5 = tindex.search(q5, num_results=5)

vec_files = [r["filename"] for r in vec_q5]
txt_files = [r["filename"] for r in txt_q5]

print("VECTOR:", vec_files)
print("TEXT:  ", txt_files)

only_vec = [f for f in vec_files if f not in txt_files]
print("\nANSWER (Q5):", only_vec[0])

VECTOR: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']
TEXT:   ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']

ANSWER (Q5): 02-vector-search/lessons/08-pgvector.md


**Answer Q5:** `02-vector-search/lessons/08-pgvector.md`  ✅ (shows up via meaning in vector search, missed by keyword search)

## Q6. Hybrid search (Reciprocal Rank Fusion)

Combine vector + text rankings with RRF (`k = 60`). Each doc scores `1 / (k + rank)` summed across the lists it appears in.

Query: `How do I give the model access to tools?`

In [9]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [10]:
q6 = "How do I give the model access to tools?"
q6_v = embedder.encode(q6)

vector_results = vindex.search(q6_v, num_results=5)
text_results = tindex.search(q6, num_results=5)

results = rrf([vector_results, text_results])

for r in results:
    print(r["filename"], "| start:", r["start"])
print("\nANSWER (Q6):", results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md | start: 4000
01-agentic-rag/lessons/01-intro.md | start: 2000
01-agentic-rag/lessons/14-agentic-loop.md | start: 0
04-evaluation/lessons/02-ground-truth.md | start: 1000
01-agentic-rag/lessons/16-other-frameworks.md | start: 0

ANSWER (Q6): 01-agentic-rag/lessons/13-function-calling.md


**Answer Q6:** `01-agentic-rag/lessons/13-function-calling.md`  ✅ (not first in either search alone — it wins by ranking high in both)

## Summary of answers

| Q | Question | Answer |
|---|----------|--------|
| Q1 | First value `v[0]` of the query embedding | **-0.02** |
| Q2 | Cosine sim of `07-sqlitesearch-vector.md` with query | **0.37** |
| Q3 | File of highest-scoring chunk | **02-vector-search/lessons/07-sqlitesearch-vector.md** |
| Q4 | First `VectorSearch` result for the eval-metric query | **04-evaluation/lessons/05-search-metrics.md** |
| Q5 | File in vector results but not text results | **02-vector-search/lessons/08-pgvector.md** |
| Q6 | First file after RRF hybrid fusion | **01-agentic-rag/lessons/13-function-calling.md** |